# UNO Vision - main pipeline

Orchestration notebook. All logic lives in `src/`; this notebook only wires modules together.



## A. Setup

In [ ]:

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import torch
import pandas as pd

from config import (
    TEST_IMAGES, SUBMISSIONS_DIR,
    CNN_CHECKPOINT, YOLO_CHECKPOINT,
    SYMBOL_CROPS_DIR, SYMBOL_CROPS_LABELS,
)
from src.player_zones import load_zone_fracs, center_roi_frac
from src.classical_backend import detect_cards_classical
from src.yolo_backend import detect_cards_yolo
from src.card_labeler import load_cnn
from src.orchestrator import run_submission
from src.validation import evaluate_backend


PREPARE_DATA = False
TRAIN_CNN = False
TRAIN_YOLO = False
RUN_BACKEND_CLASSICAL = True
RUN_BACKEND_YOLO = True
RUN_VALIDATION = True

device = 'cuda' if torch.cuda.is_available() else 'cpu'
zone_fracs = load_zone_fracs()
center_roi = center_roi_frac(zone_fracs)
print('device:', device)
print('zones:', zone_fracs)
print('center ROI:', center_roi)

## B. Symbol crop preparation

In [ ]:
if PREPARE_DATA:
    !.venv/bin/python scripts/prepare_symbol_crops.py

## C. Symbol CNN training

In [ ]:
if TRAIN_CNN:
    !.venv/bin/python hybric_detection_utils/train_uno.py train \
        --image_dir data/symbol_crops \
        --labels data/symbol_crops/labels.csv \
        --output models/uno_symbol_cnn.pt \
        --epochs 30 --batch_size 64

## D-E. Detection backends

In [ ]:


cnn, cnn_classes = load_cnn(str(CNN_CHECKPOINT), device=device)
print('CNN classes:', cnn_classes)

## E. YOLO training (gated)

In [ ]:
if TRAIN_YOLO:
    !.venv/bin/python scripts/train_yolo.py

## K. Run submission for both backends

In [ ]:

if RUN_BACKEND_CLASSICAL:
    run_submission(
        image_dir=TEST_IMAGES,
        out_csv=SUBMISSIONS_DIR / 'classical.csv',
        detect_fn=detect_cards_classical,
        cnn=cnn, classes=cnn_classes, device=device,
    )

if RUN_BACKEND_YOLO:
    run_submission(
        image_dir=TEST_IMAGES,
        out_csv=SUBMISSIONS_DIR / 'yolo.csv',
        detect_fn=detect_cards_yolo,
        cnn=cnn, classes=cnn_classes, device=device,
    )

## Backend agreement check

In [ ]:
if RUN_BACKEND_CLASSICAL and RUN_BACKEND_YOLO:
    a = pd.read_csv(SUBMISSIONS_DIR / 'classical.csv').set_index('image_id')
    b = pd.read_csv(SUBMISSIONS_DIR / 'yolo.csv').set_index('image_id')
    same = (a == b).all(axis=1)
    print(f'Row-level agreement: {same.mean():.3f} ({same.sum()}/{len(same)})')
    for col in a.columns:
        agree = (a[col] == b[col]).mean()
        print(f'  {col}: {agree:.3f}')

## L. Validation on synthetic held-out slice

In [ ]:

if RUN_VALIDATION:
    if RUN_BACKEND_CLASSICAL:
        r = evaluate_backend(detect_fn=detect_cards_classical, cnn=cnn,
                              classes=cnn_classes, device=device)
        print('CLASSICAL:', r)
    if RUN_BACKEND_YOLO:
        r = evaluate_backend(detect_fn=detect_cards_yolo, cnn=cnn,
                              classes=cnn_classes, device=device)
        print('YOLO:', r)